библиотеки

In [1]:
import os
import random
from pathlib import Path
from collections import OrderedDict
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from tqdm import tqdm
from skimage.io import imread, imsave
from torch.utils.data import Dataset
from torchvision.transforms import Grayscale

предобработка и аугментация для словарного датасета - нужно добавить предварительное отсечение значений яркости меньше 1/10 или 1/100 от максимума на изображении

In [2]:


from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureChannelFirst,
    CropForegroundd,
    Resized,
    SpatialPadd,
    ScaleIntensityd,
    EnsureTyped,RandRotated,
    RandAffined,
    RandAdjustContrastd
)

image_size = 224


class DebugBeforePad:
    def __call__(self, data):
        d = dict(data)
        for key in ("image", "mask"):
            if key not in d:
                continue

            x = d[key]
            shape = tuple(x.shape)

            if any(s == 0 for s in shape):
                print("WARNING: zero-sized dimension!",
                    f"Patient: {d['patient']}, "
                    f"slice: {d['slice_n']}"
                      )

        return d

class BinarizeMaskd:
    def __call__(self, data):
        d = dict(data)
        d["mask"] = (d["mask"] > 0).float()
        return d
    
class CheckForegroundd:
    def __call__(self, data):
        d = dict(data)
        image = d["image"]

        n = (image > 0.04).sum().item()

        if n == 0:
            print(
                f"Patient: {d['patient']}, "
                f"slice: {d['slice_n']}",
                "WARNING: no foreground voxels",
                "shape =", tuple(image.shape),
                "min =", image.min().item(),
                "max =", image.max().item(),
            )

        return d
    
preprocess = Compose([

    # H, W -> 1, H, W для маски
    EnsureChannelFirstd(
        keys=[ "image", "mask"],
        channel_dim="no_channel",
    ),
    BinarizeMaskd(),
    CheckForegroundd(),

    # Обрезаем пустые края, сохраняя соответствие маски
    CropForegroundd(
        keys=["image", "mask"],
        source_key="image",
        select_fn= lambda x: x > 0.04,
        margin=0,
    ),

    # Уменьшаем/увеличиваем с сохранением пропорций
    Resized(
        keys=["image", "mask"],
        spatial_size=image_size,
        size_mode="longest",
        mode=("area", "nearest"),
    ),
    DebugBeforePad(),

    # Дополняем короткую сторону до 256 × 256
    SpatialPadd(
        keys=["image", "mask"],
        spatial_size=(image_size, image_size),
        method="symmetric",
        mode=("constant", "constant")       #можно применить reflect для изображения - считается, что свёртка так точнее, но тогда при выводе изображения нужно будет её убрать - странно смотрится
    ),

    # Масштабируем интенсивность изображения
     ScaleIntensityd(
        keys=["image"],
        minv=0.0,
        maxv=1.0,
    ),

    # Преобразуем результат в PyTorch-тензоры
    EnsureTyped(
        keys=["image", "mask"],
        dtype=torch.float32,
    ),
])

transform = Compose([
    #EnsureChannelFirst(channel_dim="no_channel"),
    RandRotated(
    keys=["image", "mask"],
    range_x=0.17,
    prob=0.5,
    mode=("bilinear", "nearest")),
    RandAffined(
        keys=["image", "mask"],
        prob=0.5,
        rotate_range=(0.1,),
        scale_range=(0.1,),
        mode=("bilinear", "nearest")),
    RandAffined(
    keys=("image", "mask"),
    prob=0.5,
    rotate_range=(0.1,),
    scale_range=(0.1,),
    mode=("bilinear", "nearest")),
    RandAdjustContrastd(
    keys=("image",),
    gamma=(0.7, 1.5),
    prob=0.5,)
])

/Users/iaroslava/Documents/python/попытки в IT/merenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/iaroslava/Documents/python/попытки в IT/merenv/lib/python3.14/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


создание словаря

In [3]:
class Segmentation_Dataset(Dataset):
    """Brain MRI dataset for FLAIR abnormality segmentation"""

    in_channels = 1
    out_channels = 1

    def __init__(
        self,
        images_dir,
        transform=None,
        preprocess=None,
        subset="train",
        random_sampling=True,
        seed=42):

        # загрузка изображений
        volumes = {}
        masks = {}
        print("reading images...")
        for (dirpath, dirnames, filenames) in os.walk(images_dir):
            image_slices = []
            mask_slices = []
            for filename in sorted(
                filter(lambda f: ".tif" in f, filenames),
                key=lambda x: int(x.split(".")[-2].split("_")[4]),
            ):
                filepath = os.path.join(dirpath, filename)
                if "mask" in filename:
                    mask_slices.append(imread(filepath, as_gray=True)) #список с путями
                else:
                    image_slices.append(imread(filepath, as_gray=True)) #список с путями
            if len(image_slices) > 0:
                patient_id = dirpath.split("/")[-1]
                volumes[patient_id] = np.array(image_slices[1:-1]) #нампи массив
                masks[patient_id] = np.array(mask_slices[1:-1]) 
        

        self.patients = sorted(volumes)

        random.seed(seed)
        n_val = round(len(self.patients) * 0.2)
        validation_patients = random.sample(self.patients, k=n_val)
        if subset == 'train':   
            self.patients = sorted(list(set(self.patients).difference(validation_patients)))
        elif subset == 'val':
            patients = validation_patients
            self.patients = sorted(list(set(patients)))

        print("preprocessing volumes...")
        # сохранение данных пациентов попарно (volume, mask)
        self.volumes = [(volumes[k], masks[k]) for k in self.patients]

        self.preprocess = preprocess
        
        #защита от пустых срезов
        filtered_volumes = []
        filtered_patients = []

        threshold = 0.04

        for patient_id, (v, m) in zip(self.patients, self.volumes):
            # Проверяем каждый срез
            valid = np.any(v > threshold, axis=(1, 2))

            # Сохраняем только непустые срезы
            v_filtered = v[valid]
            m_filtered = m[valid]


            filtered_volumes.append((v_filtered, m_filtered))
            filtered_patients.append(patient_id)

        self.volumes = filtered_volumes
        self.patients = filtered_patients

        # probabilities for sampling slices based on masks
        self.slice_weights = [m.sum(axis=-1).sum(axis=-1) for v, m in self.volumes]
        self.slice_weights = [(s + (s.sum() * 0.1 / len(s))) / (s.sum() * 1.1) for s in self.slice_weights]

        # добавление оси канала. Это позволяет позже переставить ось канала на первое место и получить формат, ожидаемый PyTorch
        #self.volumes = [(v, m[..., np.newaxis]) for (v, m) in self.volumes]

        print("done creating dataset")

        # create global index for patient and slice (idx -> (p_idx, s_idx))
        num_slices = [v.shape[0] for v, m in self.volumes]
        self.patient_slice_index = list(
            zip(
                sum([[i] * num_slices[i] for i in range(len(num_slices))], []),
                sum([list(range(x)) for x in num_slices], []),
            ))

        self.random_sampling = random_sampling

        self.transform = transform

    def __len__(self):
        return len(self.patient_slice_index)
    
    

    def __getitem__(self, idx):
        patient = self.patient_slice_index[idx][0]
        slice_n = self.patient_slice_index[idx][1]

        if self.random_sampling:
            patient = np.random.randint(len(self.volumes))
            slice_n = np.random.choice(
                range(self.volumes[patient][0].shape[0]), p=self.slice_weights[patient]
            )

        #выбор среза из 3Д объёма
        v, m = self.volumes[patient]
        image = v[slice_n]
        mask = m[slice_n]

        data = {"image": image,
                "mask": mask,
                "patient": self.patients[patient],
                "slice_n": slice_n
                }

        if self.preprocess is not None:
            data = self.preprocess(data)


        #если заданы функции трансформации (вроде аугмнтации), применение оных
        if self.transform is not None:
            data = self.transform(data)

        image = data['image']
        mask = data['mask']

        # перестановка осей (C, H, W)
        #image = image.transpose(2, 0, 1)
        #mask = mask.transpose(2, 0, 1)

        #перевод в тензоры и преобразование формата данных
        image_tensor = torch.from_numpy(image.astype(np.float32))
        mask_tensor = torch.from_numpy(mask.astype(np.float32))

        # return tensors
        return image_tensor, mask_tensor

создание модели UNet - предобученная

In [4]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
    activation="sigmoid",
)

In [5]:
print(next(model.parameters()).device)

cpu


расположение предобученных весов

In [6]:
import torch
import os

cache_dir = torch.hub.get_dir()
print("Каталог PyTorch:", cache_dir)

checkpoint_dir = os.path.join(cache_dir, "checkpoints")
print("Каталог весов:", checkpoint_dir)

if os.path.exists(checkpoint_dir):
    print(os.listdir(checkpoint_dir))

Каталог PyTorch: /Users/iaroslava/.cache/torch/hub
Каталог весов: /Users/iaroslava/.cache/torch/hub/checkpoints
['resnet18-f37072fd.pth']


функция потерь - сравнить с ДайсЛоссом МОНАИ и выбрать, что удобнее

In [7]:
class DiceLoss(nn.Module):

    def __init__(self):
        super(DiceLoss, self).__init__()
        self.smooth = 1.0

    def forward(self, y_pred, y_true):
        assert y_pred.size() == y_true.size()
        y_pred = y_pred[:, 0].contiguous().view(-1)
        y_true = y_true[:, 0].contiguous().view(-1)
        intersection = (y_pred * y_true).sum()
        dsc = (2. * intersection + self.smooth) / (
            y_pred.sum() + y_true.sum() + self.smooth
        )
        return 1. - dsc

визуализация с масками

In [8]:
def log_images(x, y_true, y_pred, channel=1):
    images = []
    x_np = x[:, channel].detach().cpu().numpy() #перенесли на цпу и преобразовали в нампи, detach() - защита от градиентов
    y_true_np = y_true[:, 0].cpu().numpy()
    y_pred_np = y_pred[:, 0].cpu().numpy()
    image = gray2rgb(np.squeeze(x_np[i]))
    for i in range(x_np.shape[0]):
        image = outline(image, y_pred_np[i], color=[255, 0, 0])  #красный - предсказанный
        image = outline(image, y_true_np[i], color=[0, 255, 0])  #зелёный - правильный
        images.append(image)
    return images

def gray2rgb(image):  #перевод в RGB
    w, h = image.shape
    image += np.abs(np.min(image))  #сдвиг: прибавляем модуль минимальнтго значения пикселя ко всем пикселям
    #image = image - np.min(image) #сдвиг: вычитаем минимум
    image_max = np.abs(np.max(image))  #нормализация - находим максимум после сдвига
    if image_max > 0:
        image /= image_max #делим значения всех пикселей на максимум
    ret = np.empty((w, h, 3), dtype=np.uint8)  #создаём пустой массив для RGB канала
    ret[:, :, 2] = ret[:, :, 1] = ret[:, :, 0] = image * 255  #копирование изображения в три канала
    return ret

def outline(image, mask, color):  #разлиновка границ масок
    mask = np.round(mask)
    yy, xx = np.nonzero(mask) #выписываем ненулевые значения
    for y, x in zip(yy, xx):   #соезиняем их попарно
        if 0.0 < np.mean(mask[max(0, y - 1) : y + 2, max(0, x - 1) : x + 2]) < 1.0:
            image[max(0, y) : y + 1, max(0, x) : x + 1] = color
    return image


загрузка данных - проверить соответсвие названий датасетов, аргументов и аугментации(особенно внимательно)!!

In [9]:
def worker_init(worker_id):
        np.random.seed(42 + worker_id)

def data_loaders(batch_size, workers):
    dataset_train, dataset_valid = datasets("/Users/iaroslava/Documents/python/попытки в IT/data_seg")


    loader_train = DataLoader(
        dataset_train,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=workers,
        worker_init_fn=worker_init,
    )
    loader_valid = DataLoader(
        dataset_valid,
        batch_size=batch_size,
        drop_last=False,
        num_workers=workers,
        worker_init_fn=worker_init,
    )

    return loader_train, loader_valid

метрики потерь, постобработка
посмотреть оптимизацию через MONAI!!! - я не уверена, насколько предлагаемые там метрики нам подходят

In [10]:
def datasets(images_dir):
    train = Segmentation_Dataset(
        images_dir=images_dir,
        transform=transform,
        preprocess=preprocess,
        subset='train'
    )
    valid = Segmentation_Dataset(
        images_dir=images_dir,
        transform=transform,
        preprocess=preprocess,
        subset="val",
        random_sampling=False,
    )
    return train, valid


def dsc(y_pred, y_true):   #вычисление расхождения реальной маски с прогнозируенмой
    y_pred = np.round(y_pred).astype(int)   #перевод предсказания в бинарную маску
    y_true = np.round(y_true).astype(int)
    return np.sum(y_pred[y_true == 1]) * 2.0 / (np.sum(y_pred) + np.sum(y_true))  #вычисление коэффициента совпадения


def dsc_distribution(volumes):      #словарь дайслосса для каждого пациента
    dsc_dict = {}                   #Dice по совокупности пикселей всех срезов, а не среднее значение Dice по каждому срезу
    for p in volumes:
        y_pred = volumes[p][1]
        y_true = volumes[p][2]
        dsc_dict[p] = dsc(y_pred, y_true)
    return dsc_dict


def dsc_per_volume(validation_pred, validation_true, patient_slice_index): 
    dsc_list = []
    num_slices = np.bincount([p[0] for p in patient_slice_index])   #вычисление количества срезов
    index = 0
    for p in range(len(num_slices)):    #перебирание всех срезов
        y_pred = np.array(validation_pred[index : index + num_slices[p]])
        y_true = np.array(validation_true[index : index + num_slices[p]])
        dsc_list.append(dsc(y_pred, y_true))    #сохранение дайслосса для каждого среза
        index += num_slices[p]                  #вычисляется один Dice по всем срезам пациента и добавляется в список
    return dsc_list


def postprocess_per_volume(     #подготовка данных для анализа и визуализации
    input_list, pred_list, true_list, patient_slice_index, patients
):
    volumes = {}
    num_slices = np.bincount([p[0] for p in patient_slice_index])
    index = 0
    for p in range(len(num_slices)):
        volume_in = np.array(input_list[index : index + num_slices[p]]) #извлечение исходных изображений
        volume_pred = np.round(
            np.array(pred_list[index : index + num_slices[p]])
        ).astype(int)
        volume_true = np.array(true_list[index : index + num_slices[p]])
        volumes[patients[p]] = (volume_in, volume_pred, volume_true)    #сохранение данных в трёхэлементный кортеж
        index += num_slices[p]
    return volumes


def log_loss_summary(loss, step, prefix=""):    #средня функция потерь
    print("epoch {} | {}: {}".format(step, prefix + "loss", np.mean(loss)))
    


def log_scalar_summary(tag, value, step):
    print("epoch {} | {}: {}".format(step, tag, value))


def plot_dsc(dsc_dist): #график дайслосса по пациентам
    y_positions = np.arange(len(dsc_dist))
    dsc_dist = sorted(dsc_dist.items(), key=lambda x: x[1])
    values = [x[1] for x in dsc_dist]
    labels = [x[0] for x in dsc_dist]
    labels = ["_".join(l.split("_")[1:-1]) for l in labels]
    fig = plt.figure(figsize=(12, 8))
    canvas = FigureCanvasAgg(fig)
    plt.barh(y_positions, values, align="center", color="skyblue")
    plt.yticks(y_positions, labels)
    plt.xticks(np.arange(0.0, 1.0, 0.1))
    plt.xlim([0.0, 1.0])
    plt.gca().axvline(np.mean(values), color="tomato", linewidth=2)
    plt.gca().axvline(np.median(values), color="forestgreen", linewidth=2)
    plt.xlabel("Dice coefficient", fontsize="x-large")
    plt.gca().xaxis.grid(color="silver", alpha=0.5, linestyle="--", linewidth=1)
    plt.tight_layout()
    canvas.draw()
    plt.close()
    s, (width, height) = canvas.print_to_buffer()
    return np.frombuffer(s, dtype=np.uint8).reshape((height, width, 4))

In [11]:
batch_size = 16
epochs = 100
lr = 0.0001      
workers = 0
weights = '/Users/iaroslava/Documents/python/попытки в IT/data_seg/weights'
images_dir = "/Users/iaroslava/Documents/python/попытки в IT/data_seg"
image_size = 224
start_epoch = 0
output_dir = "/Users/iaroslava/Documents/python/попытки в IT/data_seg/output_dir"

сохранение

In [12]:
def checkpoint(model, loss_valid, best_validation_dsc, best_epoch, optimizer):
    print('saving')
    print()
    state = {'model': model,'best_val_loss': np.mean(loss_valid), 'best_dsc': best_validation_dsc, 'epoch': best_epoch,'rng_state': torch.get_rng_state(), 'optimizer': optimizer.state_dict(),}
    torch.save(state, os.path.join(weights,'checkpoint.pth'))
    torch.save(model.state_dict(), os.path.join(weights, 'model_weights_pt'))

основной цикл обучения

In [ ]:
def train_validate(model):
    #device = torch.device("cpu" if not torch.cuda.is_available() else "cuda:0")    #для винды
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")    #для мак

    loader_train, loader_valid = data_loaders(batch_size, workers)
    loaders = {"train": loader_train, "valid": loader_valid}

    model = model.to(device)

    dsc_loss = DiceLoss()
    best_validation_dsc = 0.0
    start_epoch = -1

    optimizer = optim.Adam(model.parameters(), lr=lr)

    checkpoint_path = Path(weights) / "checkpoint.pth"
    weights_path = Path(weights) / 'model_weights_pt'

    if checkpoint_path.exists():
        print('загружаю веса из прдыдущего обучения')
        checkpoint_data = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False
            )
        start_epoch = checkpoint_data['epoch']

        state_dict = torch.load(
            weights_path,
            map_location=device,
            weights_only=True
        )
        model.load_state_dict(state_dict)
        print("Загружены существующие веса")
    else:
        print("Сохранение не найдено. Начинаю с нуля.")

    loss_train = []
    loss_valid = []
    all_train_losses = []       #данные для графиков
    all_val_losses = []
    mean_dsc_all = []

    step = 1
    #начало эпохального цикла
    for epoch in range(start_epoch + 1, epochs):
        for phase in ["train", "valid"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            validation_pred = []
            validation_true = []

            for i, data in enumerate(loaders[phase]):
                if phase == "train":
                    step += 1

                x, y_true = data    #загрузка данных в одно место с моделью
                x, y_true = x.to(device), y_true.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    y_pred = model(x)

                    loss = dsc_loss(y_pred, y_true)

                    if phase == "valid":
                        loss_valid.append(loss.item())
                        y_pred_np = y_pred.detach().cpu().numpy()
                        validation_pred.extend(     #извлечение всех элементов батча в список
                            [y_pred_np[s] for s in range(y_pred_np.shape[0])]
                        )
                        y_true_np = y_true.detach().cpu().numpy()
                        validation_true.extend(
                            [y_true_np[s] for s in range(y_true_np.shape[0])]
                        )

                    if phase == "train":
                        loss_train.append(loss.item())  #добавление ф потерь в список - потом усредняются
                        loss.backward()     #вычисление градиентов
                        optimizer.step()        #оптимизатор

            #вывод потерь после фазы
            if phase == "train":
                log_loss_summary(loss_train, epoch)
                all_train_losses.append(np.mean(loss_train))
                loss_train = []     #обнуление списка

            if phase == "valid":
                log_loss_summary(loss_valid, epoch, prefix="val_")
                all_val_losses.append(np.mean(loss_valid))
                mean_dsc = np.mean(     #вычисление среднего дайса по валидации для каждого пациента (np.mean - среднее)
                    dsc_per_volume(
                        validation_pred,
                        validation_true,
                        loader_valid.dataset.patient_slice_index,
                    )
                )
                mean_dsc_all.append(mean_dsc)
                log_scalar_summary("val_dsc", mean_dsc, epoch)      #сохранение лучшей эпохи
                if mean_dsc > best_validation_dsc:
                    best_validation_dsc = mean_dsc
                    best_epoch = epoch
                    checkpoint(model, loss_valid, best_validation_dsc, best_epoch, optimizer)
                loss_valid = []     #обнуление списка

        if ((epoch - best_epoch) >= 10):
            print("no improvement in 10 epochs, break")
            break

    #после обучения
    print("\nBest validation mean DSC: {:4f}\n".format(best_validation_dsc))

    state_dict = torch.load(os.path.join(weights, 'model_weights_pt'))
    model.load_state_dict(state_dict)


    model.eval()
    #финальная валидация и сохранение
    input_list = []
    pred_list = []
    true_list = []

    for i, data in enumerate(loader_valid):
        x, y_true = data
        x, y_true = x.to(device), y_true.to(device)
        with torch.set_grad_enabled(False):
            x = x.squeeze(2)
            y_true = y_true.squeeze(2)
            y_pred = model(x)
            y_pred_np = y_pred.detach().cpu().numpy()
            pred_list.extend([y_pred_np[s] for s in range(y_pred_np.shape[0])])
            y_true_np = y_true.detach().cpu().numpy()
            true_list.extend([y_true_np[s] for s in range(y_true_np.shape[0])])
            x_np = x.detach().cpu().numpy()
            input_list.extend([x_np[s] for s in range(x_np.shape[0])])

    volumes = postprocess_per_volume(
        input_list,
        pred_list,
        true_list,
        loader_valid.dataset.patient_slice_index,
        loader_valid.dataset.patients,
    )

    dsc_dist = dsc_distribution(volumes)

    dsc_dist_plot = plot_dsc(dsc_dist)
    imsave("./dsc.png", dsc_dist_plot)

    for p in volumes:
        x = volumes[p][0]
        y_pred = volumes[p][1]
        y_true = volumes[p][2]
        for s in range(x.shape[0]):
            image = gray2rgb(x[s, 0])  # channel 1 is for FLAIR
            image = outline(image, y_pred[s, 0], color=[255, 0, 0])
            image = outline(image, y_true[s, 0], color=[0, 255, 0])
            filename = "{}-{}.png".format(p, str(s).zfill(2))
            filepath = os.path.join(output_dir, filename)
            imsave(filepath, image)


train_validate(model)

reading images...
preprocessing volumes...
done creating dataset
reading images...
preprocessing volumes...
done creating dataset
Сохранение не найдено. Начинаю с нуля.
epoch 0 | loss: 0.787236329290893
epoch 0 | val_loss: 0.8969763745652869
epoch 0 | val_dsc: 0.4962076909886518
saving

epoch 1 | loss: 0.5731452647801284
epoch 1 | val_loss: 0.7942326538106228
epoch 1 | val_dsc: 0.7074135081524315
saving

epoch 2 | loss: 0.34459676552604845
epoch 2 | val_loss: 0.6982588095867888
epoch 2 | val_dsc: 0.7304640357675768
saving

epoch 3 | loss: 0.20427071851688428
epoch 3 | val_loss: 0.5873328345887204
epoch 3 | val_dsc: 0.773774124904142
saving

epoch 4 | loss: 0.1371561796455593
epoch 4 | val_loss: 0.527543453460044
epoch 4 | val_dsc: 0.7605084222139668
epoch 5 | loss: 0.10447025168073046
epoch 5 | val_loss: 0.5070455771811465
epoch 5 | val_dsc: 0.7563954220478305
epoch 6 | loss: 0.08919857491503705
epoch 6 | val_loss: 0.48141615821960126
epoch 6 | val_dsc: 0.7587428297444178
epoch 7 | los